# 03 · Fully-differential amplifier: R-NSGA-II optimization with novaad + ngspice in the loop

```
                 VDD ─────────┬───────────────────┬──────────
                              │                   │
          cmfb ──────────┤ M3            M4 ├──────── cmfb
                              │                   │
              outn ───────────┼──┬─/\/\─┬─/\/\─┬──┼─────────── outp
                              │  │ R_CM │ R_CM │  │
                              │  │      cmfb   │  │
              inp ───────┤ M1 │  CL           CL  │ M2 ├───── inn
                              └────────┬──────────┘
                                     tail
   IREF ↓                              │
        nb ──┬──────────────────┤ M5  (active tail current source)
       M6 ├──┘ (diode)                 │
            GND ──────────────────────GND
```

* **M1/M2**: NMOS differential pair.
* **M3/M4**: PMOS active loads. Their gates sit at `cmfb`, the midpoint of the two `R_CM` resistors between the outputs. For the common mode the loads act as diodes, which sets the output CM to `VDD − Vsg`. For the differential signal `cmfb` is a virtual ground, so the loads act as current sources and the gain is `gm1·(ro1 ∥ ro3 ∥ R_CM)`.
* **M5/M6**: an NMOS current mirror, the *active source* of the tail current `I_TAIL`.

For each candidate design, **novaad** sizes every transistor from gm/id, L and current, and **ngspice** measures it (OP, AC, noise). **pymoo's R-NSGA-II** then searches the trade-off between four objectives:

| objective | definition |
|---|---|
| minimize **spec error** | RMS of the relative errors: `measured / target − 1` for each `SPEC_TARGETS` entry (gain, GBW, output CM, power dissipation), and the excess `max(0, measured / limit − 1)` for each `SPEC_LIMITS` entry (input-referred noise) |
| minimize **area** | `Σ W·L` of M1–M6 + area of the two poly resistors `2·(R_CM/R□)·W_res²` |
| minimize **power** | `VDD·I(VDD)` |
| maximize **Schreier FoM** | `FoM_S = SNR_dB + 10·log10(GBW / P)` |

`SNR = (A²/2) / Vn²`, where:
- `A` is the peak differential output swing allowed by the headroom of M1 and M3 at the simulated operating point.
- `Vn²` is the differential output noise integrated from `NOISE_FMIN` to `GBW`.

Maximizing `FoM_S` rewards both SNR and GBW per watt.

The specs are **targets**, not bounds: every measured parameter should land as close as possible to its target, so the combined spec error is minimized together with area, power and −FoM. There are **no hard constraints**: the result is a ranked list of the `N_CLOSEST` designs closest to the targets, and every design reports its saturation margin so devices that leave saturation are visible.

**R-NSGA-II** (reference-point NSGA-II) is NSGA-II with designer preferences. Instead of spreading solutions over the whole Pareto front, it concentrates them around **reference points**: aspiration values for `(spec error, area, power, −FoM)` set in `REF_POINTS`. `EPSILON` controls how tightly they cluster.

The Pareto front is plotted **live while the optimizer runs**:
- R-NSGA-II runs in a background thread.
- A second thread pushes the current front, with the reference points marked, into a Bokeh figure.

Run `01_pdk_ngspice_lut_extraction.ipynb` first to create the LUTs.

In [1]:
# ── Operating conditions: edit these ────────────────────────────────────────
VDD = 3.3              # supply [V]
VIN_CM = 1.2          # input common mode [V]
CL = 0.5e-12             # load capacitance per output [F]
NOISE_FMIN = 1.0       # lower bound of the noise integration [Hz] (upper bound = GBW)

# ── Spec targets: the optimizer drives each measured parameter toward its target ──
SPEC_TARGETS = {
    "gain_db": 36,        # differential DC gain [dB]
    "gbw": 500e6,        # differential gain-bandwidth product A0·f-3dB [Hz]
    "vocm": 2.0,       # output common mode [V]: set by the loads, VDD - Vsg; ~2.23 V at load gm/id = 6
    "power": 1.2e-3,    # power dissipation VDD·I(VDD) [W], including the IREF branch: low-power target
}
# one-sided limits: only exceeding the limit counts as error (noise can't usefully be pushed *up* to a target)
SPEC_LIMITS = {
    "vn_in": 20e-6,     # input-referred noise [Vrms]: output noise (NOISE_FMIN..GBW) / DC gain
}

# ── Area model ──────────────────────────────────────────────────────────────
RES_SHEET = 1000       # poly resistor sheet resistance [ohm/sq] (ppolyf_u_1k)
RES_WIDTH = 1e-6       # poly resistor width [m]

# ── Design space (bounds of the 8 optimization variables) ───────────────────
GMID_IN_BOUNDS = (12, 20)        # input pair gm/id [1/V]
GMID_LOAD_BOUNDS = (6, 18)      # load gm/id [1/V]
GMID_TAIL_BOUNDS = (6, 12)      # tail gm/id [1/V]
I_TAIL_BOUNDS = (10e-6, 500e-6) # tail current [A]
R_CM_BOUNDS = (200e3, 2e6)       # CM sensing resistors [ohm]
# channel lengths are picked from the LUT grid (L_IN, L_LOAD, L_TAIL)

# ── R-NSGA-II ───────────────────────────────────────────────────────────────
REF_POINTS = [         # aspiration points (spec error [%], area [um²], power [uW], -FoM_S [dB])
    (0, 1500, 1500, -176),  # on all targets, moderate area
    (0, 500, 1200, -174),   # on all targets, compact and lower power
]
EPSILON = 0.05         # spread of the solutions around each reference point (normalized objectives)
POP_SIZE = 48
N_GEN = 40
SEED = 1
N_WORKERS = 8          # parallel ngspice evaluations
PLOT_PERIOD = 2.0      # seconds between live plot refreshes
N_CLOSEST = 10         # length of the final list of designs closest to the targets

# ── Verification of the picked design (section 6) ───────────────────────────
TRAN_AMP = 1e-3        # differential sine amplitude [V]
# ramp and pulse levels of the differential input vd: with VIN_CM = VDD/2, vd = ±VDD swings
# each input (VIN_CM ± vd/2) over the full supply
RAMP_LOW = -VDD        # differential ramp start [V]
RAMP_HIGH = VDD        # differential ramp end [V]
STEP_LOW = -VDD        # differential pulse low level [V]
STEP_HIGH = VDD        # differential pulse high level [V]
# time scales follow the design's own -3 dB corner f3, with tau = 1 / (2*pi*f3)
TRAN_FREQ_RATIO = 0.1  # sine frequency = ratio * f3
RAMP_TAUS = 200        # ramp duration in tau: slow enough to trace the static transfer curve
STEP_TAUS = 20         # pulse window in tau: vd steps up at 10 % and back down at 50 %

NCH_LUT = "luts/gf180_nch.csv"
PCH_LUT = "luts/gf180_pch.csv"
REF_WIDTH = 10e-6

In [2]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
# the live plot embeds a srcdoc <iframe> on purpose (IFrame needs a URL)
warnings.filterwarnings("ignore", message="Consider using IPython.display.IFrame")

import shutil
from math import ceil, log10, pi
from multiprocessing.pool import ThreadPool
from threading import Event, Lock, Thread

import numpy as np
from html import escape

from IPython.display import HTML, display
from pandas import DataFrame
from bokeh.embed import file_html
from bokeh.io import output_notebook, show
from bokeh.layouts import gridplot, row
from bokeh.models import ColorBar, ColumnDataSource, LinearColorMapper
from bokeh.palettes import Viridis256
from bokeh.plotting import figure
from bokeh.resources import INLINE
from pymoo.algorithms.moo.rnsga2 import RNSGA2
from pymoo.core.callback import Callback
from pymoo.core.problem import ElementwiseProblem
from pymoo.parallelization import StarmapParallelization
from pymoo.optimize import minimize

import spice
from novaad import Device, DeviceSizingSpecification as Spec
from utils import dc_tables, dc_vectors, footprint, tran_plot

output_notebook(hide_banner=True)
nch = Device(NCH_LUT, device_type="nch", ref_width=REF_WIDTH)
pch = Device(PCH_LUT, device_type="pch", ref_width=REF_WIDTH)
LCH = sorted(float(l) for l in nch.lut["lch"].unique())
print("channel lengths in the LUT [um]:", [round(l * 1e6, 2) for l in LCH])

channel lengths in the LUT [um]: [0.28, 0.5, 1.0, 2.0, 4.0]


## 1. Sizing with novaad

A design vector is `x = [gmid_in, gmid_load, gmid_tail, iL_in, iL_load, iL_tail, i_tail, r_cm]`, where `iL_*` index into the LUT channel lengths. Bias voltages depend on each other, so a short fixed-point iteration resolves them:

* **Load**: its gate is at the output CM, so `Vsd = Vsg`. Size it, read `Vsg`, and repeat. Then `VOCM = VDD − Vsg`.
* **Input pair**: the source sits at `Vs = VIN_CM − Vgs1` with the bulk grounded, so `Vsb = Vs` and `Vds = VOCM − Vs`.
* **Tail**: `Vds = Vs`, carrying `I_TAIL`.

In [3]:
def size(x) -> dict:
    gmid_in, gmid_load, gmid_tail = x[:3]
    l_in, l_load, l_tail = (LCH[int(i)] for i in x[3:6])
    i_tail, r_cm = x[6], x[7]

    vsd = VDD / 2
    for _ in range(2):
        dcop, s_load = pch.sizing(Spec(vds=vsd, vsb=0.0, lch=l_load, gmoverid=gmid_load, ids=i_tail / 2), return_dcop=True)
        vsd = dcop.vgs[0]
    vocm = VDD - vsd

    vs = VIN_CM - 0.8
    for _ in range(2):
        dcop, s_in = nch.sizing(Spec(vds=vocm - vs, vsb=vs, lch=l_in, gmoverid=gmid_in, ids=i_tail / 2), return_dcop=True)
        vs = VIN_CM - dcop.vgs[0]

    _, s_tail = nch.sizing(Spec(vds=vs, vsb=0.0, lch=l_tail, gmoverid=gmid_tail, ids=i_tail), return_dcop=True)
    return dict(w_in=s_in.wch[0], l_in=l_in, w_load=s_load.wch[0], l_load=l_load,
                w_tail=s_tail.wch[0], l_tail=l_tail, i_tail=i_tail, r_cm=r_cm)


def area(d: dict) -> float:
    fets = 2 * d["w_in"] * d["l_in"] + 2 * d["w_load"] * d["l_load"] + 2 * d["w_tail"] * d["l_tail"]
    return fets + 2 * d["r_cm"] / RES_SHEET * RES_WIDTH ** 2


X_EXAMPLE = [12, 8, 8, 1, 1, 1, 200e-6, 200e3]
size(X_EXAMPLE)

{'w_in': 3.5753209595121764e-05,
 'l_in': 5e-07,
 'w_load': 4.9126165747640165e-05,
 'l_load': 5e-07,
 'w_tail': 3.6725029989452915e-05,
 'l_tail': 5e-07,
 'i_tail': 0.0002,
 'r_cm': 200000.0}

## 2. Testbench: OP + AC + noise in one ngspice run

The differential input comes from two E-sources around `VIN_CM`, driven by a single source `Vd`. That makes `Vd` the input of the AC transfer and the reference source of the noise analysis. `meas` finds the DC gain `A0` and the −3 dB corner `f3` of `v(outp) − v(outn)`. The noise analysis is then integrated up to `GBW = A0·f3`.

GBW is deliberately **not** read at the 0 dB crossing. Large input devices couple the input to the outputs through `Cgd`, which flattens `|H|` just above 0 dB at high frequency. The 0 dB crossing then lands far beyond the real gain-bandwidth product, and the optimizer learns to exploit that. `set sqrnoise` makes ngspice report noise in V² and V²/Hz.

`netlist(d, control, vd_source)` can swap that measurement block for any other analysis and drive `Vd` with any waveform. Section 6 uses this for the DC table, the Bode plot and the transients.

In [4]:
def fet(name, d, g, s, b, model, w, l):
    m = max(1, ceil(w / REF_WIDTH - 1e-9))   # REF_WIDTH-wide fingers, like the LUT
    return f"{name} {d} {g} {s} {b} {model} w={w / m:.6g} l={l:.6g} m={m}"


def measure() -> str:
    # control block used by the optimizer: OP + AC (finds GBW) + noise integrated up to GBW
    return f'''set sqrnoise
op
wrdata op.txt v(outp) v(outn) v(tail) i(vdd) @m1[vds] @m1[vdsat] @m3[vds] @m3[vdsat] @m5[vds] @m5[vdsat]
ac dec 20 1 100G
let gdb = db(v(outp) - v(outn))
meas ac a0 find gdb at=1
let a3 = a0 - 3
meas ac f3 when gdb=$&a3
let gbw = f3 * 10^(a0 / 20)
wrdata ac.txt gdb
noise v(outp,outn) Vd dec 20 {NOISE_FMIN} $&gbw
setplot noise1
wrdata noise_spec.txt inoise_spectrum onoise_spectrum
setplot noise2
wrdata noise_tot.txt onoise_total'''


# `control` replaces the measurement block; `vd_source` sets the differential input waveform
def netlist(d: dict, control: str | None = None, vd_source: str = "dc 0 ac 1") -> str:
    return spice.header() + f'''
* fully-differential pair, PMOS active loads, resistive CM sensing, mirrored tail
VDD vdd 0 {VDD}
IREF vdd nb {d["i_tail"]}
{fet("M6", "nb", "nb", "0", "0", "nfet_03v3", d["w_tail"], d["l_tail"])}
{fet("M5", "tail", "nb", "0", "0", "nfet_03v3", d["w_tail"], d["l_tail"])}
VCM cmi 0 {VIN_CM}
Vd vd 0 {vd_source}
Ep inp cmi vd 0 0.5
En cmi inn vd 0 0.5
{fet("M1", "outn", "inp", "tail", "0", "nfet_03v3", d["w_in"], d["l_in"])}
{fet("M2", "outp", "inn", "tail", "0", "nfet_03v3", d["w_in"], d["l_in"])}
{fet("M3", "outn", "cmfb", "vdd", "vdd", "pfet_03v3", d["w_load"], d["l_load"])}
{fet("M4", "outp", "cmfb", "vdd", "vdd", "pfet_03v3", d["w_load"], d["l_load"])}
R1 outp cmfb {d["r_cm"]}
R2 outn cmfb {d["r_cm"]}
C1 outp 0 {CL}
C2 outn 0 {CL}
.control
set wr_singlescale
set wr_vecnames
{control or measure()}
.endc
.end
'''


def simulate(d: dict, keep: bool = False) -> dict:
    wd = spice.run(netlist(d))
    op, ac = spice.read(wd / "op.txt"), spice.read(wd / "ac.txt")
    vn2 = spice.read(wd / "noise_tot.txt")["onoise_total"][0]
    gain_db, _, gbw = spice.bandwidth(ac["frequency"], ac["gdb"])
    vocm = (op["v(outp)"][0] + op["v(outn)"][0]) / 2
    # peak differential swing: each output may move until M3 (top) or M1 (bottom) leaves saturation
    swing = 2 * min(VDD - vocm - abs(op["m3.vdsat"][0]), vocm - op["v(tail)"][0] - op["m1.vdsat"][0])
    power = VDD * abs(op["i(vdd)"][0])
    snr = 10 * log10(swing ** 2 / 2 / vn2) if swing > 0 else -100.0
    r = dict(gain_db=gain_db, gbw=gbw, vocm=vocm, power=power, vn2=vn2, swing=swing, snr=snr,
             vn_in=float(np.sqrt(vn2)) / 10 ** (gain_db / 20),   # input-referred noise [Vrms]
             fom=snr + 10 * log10(gbw / power), area=area(d),
             sat=min(abs(op[f"{m}.vds"][0]) - abs(op[f"{m}.vdsat"][0]) for m in ("m1", "m3", "m5")))
    if keep:
        r["dir"] = wd
    else:
        shutil.rmtree(wd)
    return r


simulate(size(X_EXAMPLE))

{'gain_db': 37.0979727,
 'gbw': np.float64(315023124.8018857),
 'vocm': np.float64(2.30684312),
 'power': np.float64(0.0012827917211999998),
 'vn2': np.float64(5.44630143e-06),
 'swing': np.float64(1.5687016839999999),
 'snr': 53.53949055812846,
 'vn_in': 3.2595091313049206e-05,
 'fom': 167.4413534244569,
 'area': 5.216044053322149e-10,
 'sat': np.float64(0.119836316)}

## 3. The optimization problem

pymoo minimizes, so the objectives are `[spec error %, area µm², power µW, −FoM_S dB]`.

* **Spec error:** the RMS of the relative errors.
  * Targets (`SPEC_TARGETS`) are two-sided: `e_k = measured_k / target_k − 1`.
  * Limits (`SPEC_LIMITS`) are one-sided: `e_k = max(0, measured_k / limit_k − 1)`, so beating a limit costs nothing.
  * A design that hits every target and respects every limit scores 0 %.
* **No hard constraints:** every simulated design competes on the four objectives alone.

A candidate that novaad can't size (target outside the LUT) or that ngspice can't solve gets the worst objectives, so R-NSGA-II discards it.

In [5]:
XL = [GMID_IN_BOUNDS[0], GMID_LOAD_BOUNDS[0], GMID_TAIL_BOUNDS[0], 0, 0, 0, I_TAIL_BOUNDS[0], R_CM_BOUNDS[0]]
XU = [GMID_IN_BOUNDS[1], GMID_LOAD_BOUNDS[1], GMID_TAIL_BOUNDS[1]] + [len(LCH) - 1e-6] * 3 + [I_TAIL_BOUNDS[1], R_CM_BOUNDS[1]]
# display name and scale of each target, for the tables
SPEC_UNITS = {"gain_db": ("Differential gain [dB]", 1), "gbw": ("GBW [MHz]", 1e-6),
              "vocm": ("Output CM [V]", 1), "vn_in": ("Input-referred noise [uVrms]", 1e6),
              "power": ("Power dissipation [uW]", 1e6)}
FAILED = dict(F=[1e3, 1e6, 1e6, 1e3])


def spec_errors(r) -> dict:
    # relative error of each measured parameter: two-sided for targets, excess only for limits
    errors = {k: r[k] / target - 1 for k, target in SPEC_TARGETS.items()}
    errors.update({k: max(0.0, r[k] / limit - 1) for k, limit in SPEC_LIMITS.items()})
    return errors


def spec_error(r) -> float:
    # combined objective: RMS of the relative errors
    return float(np.sqrt(np.mean(np.square(list(spec_errors(r).values())))))


class FullyDifferentialAmp(ElementwiseProblem):
    def __init__(self, **kwargs):
        super().__init__(n_var=8, n_obj=4, xl=XL, xu=XU, **kwargs)

    def _evaluate(self, x, out, *args, **kwargs):
        try:
            r = simulate(size(x))
        except Exception:
            out.update(FAILED)
            return
        out["F"] = [spec_error(r) * 100, r["area"] * 1e12, r["power"] * 1e6, -r["fom"]]   # %, um^2, uW, -dB

## 4. Run R-NSGA-II with a live Pareto front

Three pieces run concurrently:
- **Optimizer thread:** `minimize(...)` evaluates each generation through a `ThreadPool`. ngspice runs as subprocesses, so threads give real parallelism.
- **Callback:** after every generation it copies the current non-dominated set into a shared `state` dict, guarded by a `Lock`.
- **Plot thread:** every `PLOT_PERIOD` seconds it reads `state`, renders the front as a self-contained Bokeh page inside an `<iframe>`, and replaces the figure below through an IPython display handle (`display_id` / `update`).

This works in Jupyter, JupyterLab and VS Code. Bokeh's own `push_notebook` needs a Jupyter comm channel that VS Code doesn't provide. The red crosses are the `REF_POINTS`.

This cell returns immediately and the figure keeps updating. The next cell waits for the optimizer to finish.

In [6]:
state = {"gen": 0, "F": np.empty((0, 4)), "best_err": float("nan"), "res": None, "error": None}
lock = Lock()
done = Event()


class Snapshot(Callback):
    def notify(self, algorithm):
        with lock:
            state["gen"] = algorithm.n_gen
            state["F"] = algorithm.opt.get("F").copy()
            state["best_err"] = float(algorithm.pop.get("F")[:, 0].min())


def optimize():
    try:
        with ThreadPool(N_WORKERS) as pool:
            problem = FullyDifferentialAmp(elementwise_runner=StarmapParallelization(pool.starmap))
            algorithm = RNSGA2(ref_points=np.array(REF_POINTS, dtype=float), epsilon=EPSILON, pop_size=POP_SIZE)
            state["res"] = minimize(problem, algorithm, ("n_gen", N_GEN), seed=SEED, callback=Snapshot(), verbose=False)
    except Exception as e:
        state["error"] = e
    finally:
        done.set()


def front_plot(F, title: str, closest=None):
    """Area vs power colored by FoM, and FoM vs spec error. F rows: [spec error %, area, power, -FoM]."""
    F = np.atleast_2d(F)
    data = dict(err=F[:, 0], area=F[:, 1], power=F[:, 2], fom=-F[:, 3])
    refs = dict(err=[p[0] for p in REF_POINTS], area=[p[1] for p in REF_POINTS],
                power=[p[2] for p in REF_POINTS], fom=[-p[3] for p in REF_POINTS])
    cmap = LinearColorMapper(palette=Viridis256, low=float(data["fom"].min()), high=float(data["fom"].max()))
    tips = [("spec error", "@err{0.0} %"), ("FoM", "@fom{0.0} dB"), ("power", "@power{0} uW"), ("area", "@area{0} um²")]
    p_ap = figure(title=title, x_axis_label="power [uW]", y_axis_label="area [um²]",
                  x_axis_type="log", y_axis_type="log", width=500, height=380, tooltips=tips)
    p_fp = figure(title="FoM vs spec error", x_axis_label="spec error, RMS [%]", y_axis_label="FoM_S [dB]",
                  width=420, height=380, tooltips=tips)
    for p, (x, y) in ((p_ap, ("power", "area")), (p_fp, ("err", "fom"))):
        p.scatter(x, y, source=ColumnDataSource(data), size=8, color={"field": "fom", "transform": cmap})
        p.scatter(x, y, source=ColumnDataSource(refs), marker="x", size=14, line_width=3, color="red")
        if closest is not None:
            c = np.atleast_2d(closest)
            near = dict(err=c[:, 0], area=c[:, 1], power=c[:, 2], fom=-c[:, 3])
            p.scatter(x, y, source=ColumnDataSource(near), size=14, fill_alpha=0, line_color="red", line_width=2)
    p_ap.add_layout(ColorBar(color_mapper=cmap, title="FoM_S [dB]"), "right")
    return row(p_ap, p_fp)


def as_iframe(layout, title: str) -> HTML:
    # a standalone page (BokehJS inlined) in an iframe renders the same in Jupyter, JupyterLab and VS Code
    page = file_html(layout, INLINE, title)
    return HTML(f'<iframe srcdoc="{escape(page)}" style="width:1000px; height:410px; border:0"></iframe>')


handle = display(HTML("<i>Pareto front: waiting for generation 1…</i>"), display_id=True)


def live_plot():
    last_gen = -1
    while True:
        finished = done.wait(PLOT_PERIOD)
        with lock:
            gen, F, best_err = state["gen"], state["F"], state["best_err"]
        if gen != last_gen and len(F):
            title = f"Generation {gen}/{N_GEN}: {len(F)} non-dominated, best spec error {best_err:.1f} %"
            handle.update(as_iframe(front_plot(F, title), title))
            last_gen = gen
        if finished:
            return


opt_thread = Thread(target=optimize, name="rnsga2", daemon=True)
plot_thread = Thread(target=live_plot, name="live-plot", daemon=True)
opt_thread.start()
plot_thread.start()

In [7]:
# wait for the optimizer (and the last plot refresh) before analysing the result
opt_thread.join()
plot_thread.join()
if state["error"]:
    raise state["error"]
res = state["res"]
print(f"{res.algorithm.evaluator.n_eval} evaluations in {res.exec_time:.1f} s, "
      f"{len(res.F)} non-dominated designs, best spec error {res.pop.get('F')[:, 0].min():.2f} %")

1920 evaluations in 52.9 s, 48 non-dominated designs, best spec error 10.62 %


## 5. Closest designs

The final Pareto front (the live figure above keeps its last frame), followed by the `N_CLOSEST` designs closest to the spec targets, ranked by spec error (ties broken by FoM). The closest designs are circled in red on the front.

* **Where they come from:** the whole final population, not only the non-dominated front. A design can be close to the targets and still be dominated on area or power.
* **Error columns:** each `err` column is one spec's relative error, two-sided for targets and excess-only for limits.
* **`sat margin`:** the smallest `|Vds| − |Vdsat|` among the transistors. A negative value means a device is out of saturation.

In [8]:
X = res.pop.get("X")
rows = []
for i, x in enumerate(X):
    try:
        rows.append({"i": i, **size(x), **simulate(size(x))})
    except Exception:
        pass   # novaad could not size it or ngspice failed: it can't be a close design anyway
designs = DataFrame(rows).set_index("i")
designs["spec_error"] = designs.apply(spec_error, axis=1)
table = DataFrame({
    "spec error [%]": designs["spec_error"] * 100,
    **{f"err {SPEC_UNITS.get(k, (k, 1))[0].split(' [')[0]} [%]": designs.apply(lambda r: spec_errors(r)[k], axis=1) * 100
       for k in {**SPEC_TARGETS, **SPEC_LIMITS}},
    "FoM_S [dB]": designs["fom"],
    "power [uW]": designs["power"] * 1e6,
    "area [um²]": designs["area"] * 1e12,
    "gain [dB]": designs["gain_db"],
    "GBW [MHz]": designs["gbw"] / 1e6,
    "VOCM [V]": designs["vocm"],
    "noise in [uVrms]": designs["vn_in"] * 1e6,
    "SNR [dB]": designs["snr"],
    "sat margin [V]": designs["sat"],
    "gm/id in": X[designs.index, 0], "gm/id load": X[designs.index, 1], "gm/id tail": X[designs.index, 2],
    "L in [um]": designs["l_in"] * 1e6, "L load [um]": designs["l_load"] * 1e6, "L tail [um]": designs["l_tail"] * 1e6,
    "W in [um]": designs["w_in"] * 1e6, "W load [um]": designs["w_load"] * 1e6, "W tail [um]": designs["w_tail"] * 1e6,
    "I tail [uA]": designs["i_tail"] * 1e6, "R_CM [kohm]": designs["r_cm"] / 1e3,
}, index=designs.index).sort_values(["spec error [%]", "FoM_S [dB]"], ascending=[True, False])
# keep distinct designs only: neighbours that round to the same spec error, power and area are duplicates
key = DataFrame({"err": table["spec error [%]"].round(2), "p": table["power [uW]"].round(0), "a": table["area [um²]"].round(0)})
table = table[~key.duplicated()]
order = table.index[:N_CLOSEST]   # rows of X, closest first
table = table.loc[order].reset_index(drop=True)

closest = np.column_stack([table["spec error [%]"], table["area [um²]"], table["power [uW]"], -table["FoM_S [dB]"]])
show(front_plot(res.F, f"Final Pareto front: {len(res.F)} designs, {len(table)} closest circled", closest))
table.round(2)

,spec error [%],err Differential gain [%],err GBW [%],err Output CM [%],err Power dissipation [%],err Input-referred noise [%],FoM_S [dB],power [uW],area [um²],gain [dB],...,gm/id load,gm/id tail,L in [um],L load [um],L tail [um],W in [um],W load [um],W tail [um],I tail [uA],R_CM [kohm]
0,10.62,0.13,-16.39,13.84,10.19,0.00,173.33,1322.28,2014.03,36.05,...,6.05,6.47,0.5,0.28,4.0,261.81,13.63,155.78,201.05,249.17
1,10.71,-0.66,-18.16,13.88,7.14,0.00,173.49,1285.65,983.94,35.76,...,6.02,6.60,0.5,0.28,2.0,255.08,13.28,77.54,195.89,205.63
2,10.71,-0.75,-18.16,13.88,7.14,0.07,173.51,1285.65,974.40,35.73,...,6.05,6.60,0.5,0.28,2.0,255.08,13.28,77.54,195.89,200.86
3,10.72,-0.74,-18.60,13.84,5.98,0.46,173.50,1271.81,1863.40,35.73,...,6.07,6.62,0.5,0.28,4.0,251.82,13.11,149.83,193.38,202.79
4,10.73,-0.61,-15.86,13.87,11.46,0.00,173.54,1337.52,1394.91,35.78,...,6.02,9.29,0.5,0.28,2.0,265.24,13.81,180.44,203.69,200.08
5,10.75,-0.55,-15.75,13.87,11.71,0.00,173.53,1340.50,1146.04,35.80,...,6.02,8.63,0.5,0.28,2.0,265.88,13.84,116.71,204.18,202.78
6,10.76,-0.37,-17.34,13.93,9.17,0.00,173.43,1310.05,830.06,35.87,...,6.00,8.49,0.5,0.28,1.0,260.70,13.57,61.75,200.20,219.14
7,10.76,-0.54,-17.34,13.93,9.17,0.00,173.48,1310.05,809.87,35.80,...,6.00,8.54,0.5,0.28,1.0,260.70,13.57,61.75,200.20,209.04
8,10.80,-0.76,-18.88,13.88,5.77,0.50,173.50,1269.21,970.92,35.73,...,6.02,6.62,0.5,0.28,2.0,251.82,13.11,76.55,193.38,202.79
9,10.81,-0.64,-16.61,13.93,10.68,0.00,173.52,1328.13,760.29,35.77,...,6.04,7.37,0.5,0.28,1.0,264.46,13.76,42.67,203.09,201.39


## 6. Inspect one design

Pick a row of the table above (`PICK = 0` is the design closest to the spec targets).

### 6.1 DC operating point

One `op` run saves every node voltage and element current. `utils.dc_tables` maps each element's terminals to nets through InSpice's parse of the netlist.

In [9]:
PICK = 0

x = X[order[PICK]]
d = size(x)
bench = netlist(d)
op = spice.read(spice.run(netlist(d, f"op\nwrdata op.txt {dc_vectors(bench)}")) / "op.txt")
devices, nodes = dc_tables(bench, op)
display(devices.round(4).fillna(""))
nodes.round(4)

,type,nets,I [uA],V(+) [V],V(-) [V],V(d) [V],V(g) [V],V(s) [V],V(b) [V],vgs [V],vds [V],vbs [V],vth [V],vdsat [V],gm [uS],gds [uS],region,V(c+) [V],V(c-) [V]
element,,,,,,,,,,,,,,,,,,,
VDD,V,+=vdd -=0,-400.6911,3.3,0.0,,,,,,,,,,,,,,
IREF,I,+=vdd -=nb,201.0548,3.3,0.9251,,,,,,,,,,,,,,
M6,nfet_03v3,d=nb g=nb s=0 b=0,201.0548,,,0.9251,0.9251,0.0,0.0,0.9251,0.9251,0.0,0.6577,0.2301,1336.5663,2.065,saturation,,
M5,nfet_03v3,d=tail g=nb s=0 b=0,199.6236,,,0.4004,0.9251,0.0,0.0,0.9251,0.4004,0.0,0.6577,0.2301,1327.2914,4.7178,saturation,,
VCM,V,+=cmi -=0,0.0000,1.2,0.0,,,,,,,,,,,,,,
Vd,V,+=vd -=0,0.0000,0.0,0.0,,,,,,,,,,,,,,
Ep,E,+=inp -=cmi c+=vd c-=0,0.0000,1.2,1.2,,,,,,,,,,,,,0.0,0.0
En,E,+=cmi -=inn c+=vd c-=0,0.0000,1.2,1.2,,,,,,,,,,,,,0.0,0.0
M1,nfet_03v3,d=outn g=inp s=tail b=0,99.8122,,,2.2768,1.2,0.4004,0.0,0.7996,1.8764,-0.4004,0.8382,0.0735,1936.2675,10.887,saturation,,


,V [V]
node,
cmfb,2.2768
cmi,1.2000
inn,1.2000
inp,1.2000
nb,0.9251
outn,2.2768
outp,2.2768
tail,0.4004
vd,0.0000


### 6.2 AC: Bode plot of the differential gain `(v(outp) − v(outn)) / vd`

In [10]:
AC = '''ac dec 50 1 100G
let vod = v(outp) - v(outn)
let gdb = db(vod)
let gph = 180 / pi * cph(vod)
wrdata ac.txt gdb gph'''
ac = spice.read(spice.run(netlist(d, AC)) / "ac.txt")
f, mag, phase = ac["frequency"].to_numpy(), ac["gdb"].to_numpy(), ac["gph"].to_numpy()
gain_db, f3, gbw = spice.bandwidth(f, mag)
print(f"DC gain {gain_db:.2f} dB, f-3dB {f3 / 1e6:.3f} MHz, GBW = A0·f-3dB {gbw / 1e6:.1f} MHz, "
      f"|H| at GBW {np.interp(gbw, f, mag):.1f} dB")

p_mag = figure(title="Magnitude", x_axis_type="log", x_axis_label="f [Hz]", y_axis_label="|vod/vd| [dB]", width=900, height=280)
p_mag.line(f, mag, line_width=2)
p_mag.scatter([f3], [gain_db - 3], size=8, color="red")
p_mag.line([f3, gbw], [gain_db, 0], line_dash="dashed", color="gray", legend_label="single-pole asymptote to GBW")
p_ph = figure(title="Phase", x_axis_type="log", x_axis_label="f [Hz]", y_axis_label="phase [deg]", width=900, height=280, x_range=p_mag.x_range)
p_ph.line(f, phase, line_width=2)
show(gridplot([[p_mag], [p_ph]]))

DC gain 36.05 dB, f-3dB 6.601 MHz, GBW = A0·f-3dB 418.7 MHz, |H| at GBW 0.1 dB


### 6.3 Transient: sine, ramp and step

Three runs with different waveforms on the differential input `vd`. The time scales come from this design's `f−3dB`, with `τ = 1/(2π·f−3dB)`:
- **Sine:** at `TRAN_FREQ_RATIO·f−3dB`.
- **Ramp:** `vd` from `RAMP_LOW` to `RAMP_HIGH` in `RAMP_TAUS·τ`, so each input sweeps the full supply. The differential output traces the whole large-signal transfer curve and clips on both sides.
- **Step:** a pulse of `vd` from `STEP_LOW` to `STEP_HIGH` and back, again rail to rail on each input. The pair fully steers the tail current to one side, so the differential output **slews** at roughly `I_TAIL/CL`. The transistors' own capacitances lower it. With one input at the rail, the tail node rises and a short-channel tail mirror delivers more than `I_TAIL`, which raises it. The slew rate is measured as the slope between 40 % and 60 % of each transition (`spice.slew_rate`), which ignores the kick the input edge couples through `Cgd`.

Input and output share the same voltage axis, so the gain is visible directly as the ratio of the two swings.

In [11]:
tau = 1 / (2 * pi * f3)


def tran(vd_source: str, tstop: float):
    control = f"tran {tstop / 2000} {tstop}\nlet vod = v(outp) - v(outn)\nwrdata tran.txt v(vd) vod"
    return spice.read(spice.run(netlist(d, control, vd_source)) / "tran.txt")


f_sine = TRAN_FREQ_RATIO * f3
sine = tran(f"dc 0 sin(0 {TRAN_AMP} {f_sine})", 6 / f_sine)
ramp = tran(f"pwl(0 {RAMP_LOW} {RAMP_TAUS * tau} {RAMP_HIGH})", RAMP_TAUS * tau)
t_win = STEP_TAUS * tau
t_up, t_down = t_win / 10, t_win / 2
t_edge = t_win / 1000   # input rise/fall time
step = tran(f"pulse({STEP_LOW} {STEP_HIGH} {t_up} {t_edge} {t_edge} {t_down - t_up - t_edge} 1)", t_win)

last = sine["time"] >= 5 / f_sine
gain_tran_db = 20 * log10((sine["vod"][last].max() - sine["vod"][last].min()) / 2 / TRAN_AMP)
sr_up = spice.slew_rate(step["time"], step["vod"], t_up, t_down)
sr_down = spice.slew_rate(step["time"], step["vod"], t_down, t_win)
sr_expected = d["i_tail"] / CL
print(f"sine gain {gain_tran_db:.2f} dB | ramp output {ramp['vod'].min():.2f} .. {ramp['vod'].max():.2f} V | "
      f"slew rate {sr_up / 1e6:.1f} / {sr_down / 1e6:.1f} V/us up/down (bound I_TAIL/CL = {sr_expected / 1e6:.1f})")

ns = dict(t_scale=1e9, t_unit="ns", in_label="vd", out_label="vod")
show(gridplot([
    [tran_plot(f"Sine: {TRAN_AMP * 1e3:g} mV @ {f_sine / 1e6:.3g} MHz", sine["time"], sine["v(vd)"], sine["vod"], **ns)],
    [tran_plot(f"Ramp: vd {RAMP_LOW:g} V to {RAMP_HIGH:g} V in {RAMP_TAUS * tau * 1e9:.0f} ns", ramp["time"], ramp["v(vd)"], ramp["vod"], **ns)],
    [tran_plot(f"Step: vd {STEP_LOW:g} V to {STEP_HIGH:g} V and back", step["time"], step["v(vd)"], step["vod"], **ns)],
]))

sine gain 36.00 dB | ramp output -2.26 .. 2.25 V | slew rate 269.4 / 269.4 V/us up/down (bound I_TAIL/CL = 402.1)


### 6.4 Noise and spec check

The design is re-simulated with the optimizer's own measurement (`simulate`), and the files are kept so the output noise density can be plotted. The final area is also given as the side of the equivalent square footprint (`√A × √A`): transistor gate areas plus the two poly resistors, before layout overheads.

In [12]:
r = simulate(d, keep=True)
noise = spice.read(r["dir"] / "noise_spec.txt")
p = figure(title="Noise density", x_axis_type="log", y_axis_type="log", x_axis_label="f [Hz]",
           y_axis_label="nV/√Hz", width=900, height=300)
p.line(noise["frequency"], np.sqrt(noise["inoise_spectrum"]) * 1e9, line_width=2, legend_label="input-referred")
p.line(noise["frequency"], np.sqrt(noise["onoise_spectrum"]) * 1e9, line_width=2, color="orange", legend_label="output")
show(p)

errors = spec_errors(r)
report = [
    (SPEC_UNITS.get(k, (k, 1))[0], f"target {t * SPEC_UNITS.get(k, (k, 1))[1]:g}",
     f"{r[k] * SPEC_UNITS.get(k, (k, 1))[1]:.2f} ({errors[k]:+.1%})")
    for k, t in SPEC_TARGETS.items()
] + [
    (SPEC_UNITS.get(k, (k, 1))[0], f"limit <= {t * SPEC_UNITS.get(k, (k, 1))[1]:g}",
     f"{r[k] * SPEC_UNITS.get(k, (k, 1))[1]:.2f} (excess {errors[k]:.1%})")
    for k, t in SPEC_LIMITS.items()
] + [
    ("Spec error, RMS [%]", "minimize", f"{spec_error(r) * 100:.2f}"),
    # the sine sits on the pole's roll-off: compare it with the AC gain at the same frequency
    ("Transient gain [dB]", f"AC gain at {f_sine / 1e6:.3g} MHz: {np.interp(f_sine, f, mag):.2f}", f"{gain_tran_db:.2f}"),
    ("Slew rate, vod up/down [V/us]", f"reference I_TAIL/CL = {sr_expected / 1e6:.1f}", f"{sr_up / 1e6:.1f} / {sr_down / 1e6:.1f}"),
    ("Saturation margin [V]", "> 0: all devices saturated", f"{r['sat']:.2f}"),
    ("Diff. swing, peak [V]", "", f"{r['swing']:.2f}"),
    ("Output noise [uVrms]", "", f"{np.sqrt(r['vn2']) * 1e6:.2f}"),
    ("SNR [dB]", "", f"{r['snr']:.2f}"),
    ("Power [uW]", "minimize", f"{r['power'] * 1e6:.2f}"),
    ("Area [um²]", "minimize", f"{r['area'] * 1e12:.2f}"),
    ("Area, equivalent square", "", footprint(r["area"])),
    ("Schreier FoM [dB]", "maximize", f"{r['fom']:.2f}"),
]
shutil.rmtree(r["dir"])
DataFrame(report, columns=["metric", "target", "ngspice"]).set_index("metric")

,target,ngspice
metric,,
Differential gain [dB],target 36,36.05 (+0.1%)
GBW [MHz],target 500,418.05 (-16.4%)
Output CM [V],target 2,2.28 (+13.8%)
Power dissipation [uW],target 1200,1322.28 (+10.2%)
Input-referred noise [uVrms],limit <= 20,19.66 (excess 0.0%)
"Spec error, RMS [%]",minimize,10.62
Transient gain [dB],AC gain at 0.66 MHz: 36.00,36.00
"Slew rate, vod up/down [V/us]",reference I_TAIL/CL = 402.1,269.4 / 269.4
Saturation margin [V],> 0: all devices saturated,0.17
